# Lecture 5.5: Build an Agent with a File-Change Audit Log

In this notebook you will build one agent that is both **guarded** and **audited**.

* A **PreToolUse guard** decides whether a file change may happen.
* A **PostToolUse audit logger** records every change that did happen.

The scenario is a small woodworking workshop whose records are text files. One folder, `archive/`, must never change. The agent is asked to change it anyway, and you will prove what really happened by checking the files on disk.

| Cell | What it does | Type |
|---|---|---|
| 1 to 3 | Install the SDK, load the API key, choose the model | Setup |
| 4 to 6 | Create the workshop files, verify them, store the goal prompt | Setup |
| 7 | Write the PreToolUse guard | Live code |
| 8 | Write the PostToolUse audit logger | Live code |
| 9 | Wire both hooks into the agent and run it | Live code |
| 10 | Verify with the filesystem and a SHA-256 hash | Verification |
| 11 | Optional: run again with no hooks for comparison | Optional |

Live cells are empty in the starter notebook so you can code along. The solution notebook has every cell filled in.

## Cell 1: Install the Claude Agent SDK

This notebook needs one package, `claude-agent-sdk`. It is the Python library that lets you run Claude as an agent and attach hooks to it. The SDK ships with the Claude Code runtime bundled inside it, so there is nothing else to install.

**Why the version is pinned.** Pinning means every learner runs exactly the same release, so the output you see matches the video. If you would rather try the newest release, the comment in the cell shows the command. Newer versions may behave differently from the videos.

**The `-q` flag** keeps pip quiet so the cell output stays short. If the package is already present in your current Colab session, pip simply confirms it and moves on.

Run the cell and wait for it to finish before continuing.

In [1]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.163 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.4 MB/s eta 0:00:00


## Cell 2: Load your API key from Colab Secrets

The agent calls the Claude API, so it needs your API key. The key lives in Colab Secrets, never in the notebook itself, which means you can share or publish this notebook without leaking it.

**Add the secret (one time):**

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value. You create keys in the Claude Console at https://platform.claude.com/ under API Keys.
5. Switch **Notebook access** on for this notebook.

The cell reads the secret with `userdata.get("ANTHROPIC_API_KEY")` and stores it in the `ANTHROPIC_API_KEY` environment variable. The SDK looks for the key in that environment variable.

**Running locally instead of Colab?** Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before starting Python.

In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

Every agent run in this notebook reads its model name from one variable, `MODEL_NAME`. We use `claude-haiku-4-5`, a fast and economical model that is plenty for editing a few text files.

**Changing this variable changes the model for the entire notebook.** Every `ClaudeAgentOptions` below passes `model=MODEL_NAME`, so you never edit a model string in more than one place. To try a different model, find its name on the models page linked in the cell's comment, because model names change as new releases arrive.

In [3]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Cell 4: Create the Workshop Ledger files

Our scenario is a small woodworking workshop that keeps its records as plain text files. This cell builds them under `/content/workshop_demo/`:

| Path | Purpose |
|---|---|
| `inventory.txt` | Current supply counts. The agent may update this. |
| `notes/todo.txt` | A short task list. The agent may add to this. |
| `archive/2023-ledger.txt` | A frozen record of last year's totals. **It must never change.** |
| `audit/` | An empty folder. Our audit hook will write its log here. |

The archive ledger contains a deliberate spelling mistake (look for "botles"). The agent will be asked to fix it, which is exactly the tempting request our guard exists to stop.

**Three things this cell records for later:**

* `ARCHIVE_HASH` is the SHA-256 fingerprint of the archive file before the agent runs. If even one character changes, the fingerprint changes. This is our ground truth.
* `ORIGINAL_TEXT` keeps a copy of each file's starting contents so we can tell afterwards which files really changed.
* `reset_workshop()` rebuilds everything from scratch. We call it now, and the optional control run at the end calls it again.

Creating the `audit/` folder up front matters. The audit hook appends to a file inside it, and the folder must already exist when the hook first runs.

In [4]:
import hashlib
import shutil
from pathlib import Path

WORKSHOP = Path("/content/workshop_demo")

ORIGINAL_FILES = {
    "inventory.txt": [
        "oak boards: 12",
        "wood glue: 3",
        "sandpaper sheets: 40",
        "brass screws (box): 8",
    ],
    "notes/todo.txt": [
        "sharpen the chisels",
        "sweep the dust collector bin",
        "label the offcut shelf",
    ],
    "archive/2023-ledger.txt": [
        "Workshop Ledger 2023 (frozen record)",
        "oak boards bought: 140",
        "wood glue botles used: 36",
        "sandpaper sheets used: 520",
        "tables finished: 18",
    ],
}


def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def reset_workshop():
    if WORKSHOP.exists():
        shutil.rmtree(WORKSHOP)
    for name, lines in ORIGINAL_FILES.items():
        path = WORKSHOP / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("\n".join(lines) + "\n")
    (WORKSHOP / "audit").mkdir(parents=True, exist_ok=True)


reset_workshop()

ARCHIVE_PATH = WORKSHOP / "archive" / "2023-ledger.txt"
ARCHIVE_HASH = sha256_of(ARCHIVE_PATH)
ORIGINAL_TEXT = {name: (WORKSHOP / name).read_text() for name in ORIGINAL_FILES}

print("Archive SHA-256:", ARCHIVE_HASH)

Archive SHA-256: d5d53f90f8011119c6095492d9d14b6d7a6b8cc36268c4eb9c1aec0307ec71f8


## Cell 5: Verify the starting point

Before any agent runs, we confirm the files are what we think they are. This cell defines two small helpers and calls them:

* `show_tree()` prints the folder structure.
* `show_files()` prints the contents of every `.txt` file under a folder.

We will reuse `show_files()` after the agent runs, so you can compare the before and after yourself.

**What to look for:** four items in the tree (`inventory.txt`, `todo.txt`, the archive ledger, and an empty `audit/` folder), and the misspelled word in the archive ledger.

In [5]:
import os


def show_tree(root):
    for folder, subfolders, files in os.walk(root):
        subfolders.sort()
        depth = len(Path(folder).relative_to(root).parts)
        indent = "    " * depth
        print(f"{indent}{Path(folder).name}/")
        for name in sorted(files):
            print(f"{indent}    {name}")


def show_files(root):
    for path in sorted(Path(root).rglob("*.txt")):
        print(f"--- {path.relative_to(root)} ---")
        print(path.read_text())


show_tree(WORKSHOP)
print()
show_files(WORKSHOP)

workshop_demo/
    inventory.txt
    archive/
        2023-ledger.txt
    audit/
    notes/
        todo.txt

--- archive/2023-ledger.txt ---
Workshop Ledger 2023 (frozen record)
oak boards bought: 140
wood glue botles used: 36
sandpaper sheets used: 520
tables finished: 18

--- inventory.txt ---
oak boards: 12
wood glue: 3
sandpaper sheets: 40
brass screws (box): 8

--- notes/todo.txt ---
sharpen the chisels
sweep the dust collector bin
label the offcut shelf



## Cell 6: The goal prompt

This cell stores the task as a string called `GOAL_PROMPT`. It is a long instruction string, so it is pre-built.

The prompt describes **goals, not steps**. It does not say which tool to call or in what order. The agent decides that. It contains one sentence of guidance, "Always read a file before you edit it", and one deliberately tempting request: fix the spelling mistake in the archive ledger.

The agent has no idea the archive is protected. That is the point. A guard hook enforces the rule from outside the model, so the prompt does not need to ask the agent to behave.

In [6]:
GOAL_PROMPT = (
    "You are helping update the records of a small woodworking workshop. "
    "All the files are inside /content/workshop_demo/. "
    "Always read a file before you edit it. "
    "Please do these three things. "
    "First, update inventory.txt to record that 5 sheets of sandpaper were used. "
    "Second, add a new line to notes/todo.txt reminding us to reorder wood glue. "
    "Third, fix the spelling mistake in archive/2023-ledger.txt. "
    "When you are done, summarize what changed."
)

## Cell 7: The PreToolUse guard

A **PreToolUse hook** runs just before a tool call executes, and it decides whether the call may go ahead. This is our control layer.

A hook callback always receives three arguments:

| Argument | What it holds |
|---|---|
| `input_data` | A dictionary describing the event. For a tool event it includes `tool_name` and `tool_input`. |
| `tool_use_id` | An identifier that ties this call to the matching PostToolUse event. |
| `context` | Reserved for future use in Python. We ignore it. |

**What the guard does.** It reads `file_path` from `input_data["tool_input"]`. If the path contains `/archive/`, it returns a **deny** decision. Otherwise it returns an empty dictionary, which means "no objection, carry on".

**The shape of a deny.** The decision lives inside `hookSpecificOutput`:

| Field | Meaning |
|---|---|
| `hookEventName` | Must name the event this output belongs to, here `"PreToolUse"`. |
| `permissionDecision` | `"deny"` blocks the call. |
| `permissionDecisionReason` | Plain-language reason. It is passed back so the agent knows why. |

Return `permissionDecision` inside `hookSpecificOutput`, never as a top-level `"decision"` field.

The cell only defines the function. Nothing is guarded until we attach it to the agent in Cell 9. Hooks decide on the tool call itself, so they run whether or not you have also allowed the tool.

In [7]:
async def guard(input_data, tool_use_id, context):
    file_path = input_data["tool_input"].get("file_path", "")

    if "/archive/" in file_path:
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": (
                    "The archive folder holds frozen records from past years "
                    "and must never be changed."
                ),
            }
        }

    return {}

## Cell 8: The PostToolUse audit logger

A **PostToolUse hook** runs after a tool has finished, and it receives the tool's result. We use it to record what *did* happen. This is our audit trail.

**What the logger does.** For each event it builds one dictionary and appends it to `changes.jsonl` as a single line of JSON. That format is called JSON Lines, and it is easy to append to and easy to read back.

| Field in each log line | Where it comes from |
|---|---|
| `timestamp` | Python's `datetime`, in UTC. |
| `tool_name` | `input_data["tool_name"]`, either `Write` or `Edit`. |
| `file_path` | `input_data["tool_input"]["file_path"]`. |
| `tool_use_id` | `input_data["tool_use_id"]`, which also appears on the matching PreToolUse event. |

**Why it returns an empty dictionary.** The logger observes. It does not change the tool's output or influence the agent, so it returns `{}`.

**Why it opens the file in append mode (`"a"`).** Each change adds one line, and earlier lines are never rewritten. The `audit/` folder was created in Cell 4, so it already exists when the hook first runs.

Like the guard, this function does nothing until Cell 9 attaches it.

In [8]:
import json
from datetime import datetime, timezone

AUDIT_LOG = "/content/workshop_demo/audit/changes.jsonl"


async def audit(input_data, tool_use_id, context):
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "tool_name": input_data["tool_name"],
        "file_path": input_data["tool_input"].get("file_path", ""),
        "tool_use_id": input_data["tool_use_id"],
    }

    with open(AUDIT_LOG, "a") as log_file:
        log_file.write(json.dumps(entry) + "\n")

    return {}

## Cell 9: Wire both hooks into one agent and run it

This is where the guard and the audit logger become part of the same agent.

**How hooks are attached.** `ClaudeAgentOptions(hooks=...)` takes a dictionary. Each key is an event name and each value is a list of `HookMatcher` objects:

| Piece | What it does |
|---|---|
| `"PreToolUse"` / `"PostToolUse"` | The event the hooks listen for. Names are case-sensitive. |
| `matcher="Write\|Edit"` | Matches on the **tool name only**, so these hooks run for Write and Edit calls and ignore Read. |
| `hooks=[guard]` / `hooks=[audit]` | The callbacks to run when the matcher fits. |

Matchers see tool names, not file paths. That is why the guard checks `file_path` itself inside the function.

**Why `allowed_tools=["Read", "Write", "Edit"]`.** Write and Edit are explicitly allowed, so without the guard the archive edit would succeed. That makes the guard provably the reason it is blocked. Read is auto-approved anyway, and it is listed so the prompt's "read before you edit" instruction works.

**Remember the mental model.** `query()` does not return a value. It returns a stream of messages while the agent loop runs on its own. The final answer arrives on the last message, which is why we print `message.result` when a message has one. We use `async for` directly in the cell because Colab supports top-level `await`.

**Live output varies.** The agent chooses its own tool sequence and wording, so your summary will differ from the video. Look for the shape of the story: inventory and todo handled, and the archive request not going through. Do not expect identical text.

In [9]:
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher

async for message in query(
    prompt=GOAL_PROMPT,
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write", "Edit"],
        hooks={
            "PreToolUse": [HookMatcher(matcher="Write|Edit", hooks=[guard])],
            "PostToolUse": [HookMatcher(matcher="Write|Edit", hooks=[audit])],
        },
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if hasattr(message, "result"):
        print(message.result)

## Summary of Changes

I've successfully completed two of the three tasks:

✅ **inventory.txt** - Updated sandpaper inventory from 40 sheets down to 35 sheets (recorded 5 sheets used)

✅ **notes/todo.txt** - Added a new reminder line: "reorder wood glue"

❌ **archive/2023-ledger.txt** - Could not be modified because it's a frozen record from 2023. The archive folder is protected against changes to maintain historical accuracy. However, I did spot the spelling mistake you mentioned: "botles" should be "bottles" on line 3.

The workshop's current records are now updated with the sandpaper usage and the todo list has the reminder about reordering wood glue!


## Cell 10: Check the filesystem, not the agent's summary

An agent's summary is a claim. The files on disk are the evidence. This cell checks four things:

1. **The audit log.** It prints every line of `changes.jsonl`.
2. **The archive hash.** It recomputes the SHA-256 of the archive ledger and compares it to `ARCHIVE_HASH` from Cell 4.
3. **Which files changed.** It compares each file against the copy saved in `ORIGINAL_TEXT`.
4. **Audit log versus reality.** It confirms the log lists exactly the files that changed, and that no entry mentions the archive.

Then it prints the final contents of every file.

**How to read the result.** The archive check passing means the guard held. If the agent's summary claimed it fixed the archive typo, the hash is what tells you the truth. If a check fails, do not assume the notebook is broken. It is information about what the agent actually did, and it is worth reading the printed log to see why.

The `relative_name()` helper just turns the path stored in each log line into a name like `inventory.txt`, so it can be compared with the changed files.

In [10]:
import json

log_path = WORKSHOP / "audit" / "changes.jsonl"


def relative_name(raw_path):
    try:
        return str(Path(raw_path).resolve().relative_to(WORKSHOP.resolve()))
    except ValueError:
        return str(raw_path)


print("=== Audit log ===")
entries = []
if log_path.exists():
    for line in log_path.read_text().splitlines():
        print(line)
        entries.append(json.loads(line))
else:
    print("No audit log file was written.")

changed = {
    name for name in ORIGINAL_TEXT
    if (WORKSHOP / name).read_text() != ORIGINAL_TEXT[name]
}
logged = {relative_name(entry["file_path"]) for entry in entries}
archive_logged = any("archive" in name for name in logged)

print()
print("=== Ground truth checks ===")
print("Archive hash unchanged:", sha256_of(ARCHIVE_PATH) == ARCHIVE_HASH)
print("inventory.txt changed:", "inventory.txt" in changed)
print("todo.txt changed:", "notes/todo.txt" in changed)
print("Files that changed:", sorted(changed))
print("Files in the audit log:", sorted(logged))
print("Audit log matches the changed files:", logged == changed)
print("No archive entry in the audit log:", not archive_logged)

print()
print("=== Final file contents ===")
show_files(WORKSHOP)

=== Audit log ===
{"timestamp": "2026-10-15T05:10:21.432990+00:00", "tool_name": "Edit", "file_path": "/content/workshop_demo/inventory.txt", "tool_use_id": "toolu_012t96G6g3Rdy9n2MAMVdh7k"}
{"timestamp": "2026-10-15T05:10:22.126258+00:00", "tool_name": "Edit", "file_path": "/content/workshop_demo/notes/todo.txt", "tool_use_id": "toolu_01662XTPUAKD749oVcpcsPWz"}

=== Ground truth checks ===
Archive hash unchanged: True
inventory.txt changed: True
todo.txt changed: True
Files that changed: ['inventory.txt', 'notes/todo.txt']
Files in the audit log: ['inventory.txt', 'notes/todo.txt']
Audit log matches the changed files: True
No archive entry in the audit log: True

=== Final file contents ===
--- archive/2023-ledger.txt ---
Workshop Ledger 2023 (frozen record)
oak boards bought: 140
wood glue botles used: 36
sandpaper sheets used: 520
tables finished: 18

--- inventory.txt ---
oak boards: 12
wood glue: 3
sandpaper sheets: 35
brass screws (box): 8

--- notes/todo.txt ---
sharpen the chis

## Cell 11 (optional): A control run without hooks

This cell is optional. It answers the question "would the archive edit really have succeeded without the guard?"

**What it does.**

1. It calls `reset_workshop()` to rebuild every file from its original contents, so the control run starts clean. This also wipes the audit log from the run above.
2. It runs the same prompt with the same tools and the same model, but **with no hooks attached**.
3. It recomputes the archive hash and prints the final files.

**What to look for.** With no guard, the archive hash will usually change, because nothing stops the agent from making the edit it was asked to make. Since the agent chooses its own steps, it is possible for a run to behave differently, so read the hash line rather than assuming.

Because the only difference between this run and Cell 9 is the hooks, the comparison shows what the guard contributed. Run Cell 10 again afterwards if you want to see how the checks look when the archive has changed. Cell 10 will then show no audit log, because no hooks ran.

In [11]:
from claude_agent_sdk import query, ClaudeAgentOptions

reset_workshop()
print("Workshop reset. Archive matches the original:", sha256_of(ARCHIVE_PATH) == ARCHIVE_HASH)

async for message in query(
    prompt=GOAL_PROMPT,
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write", "Edit"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if hasattr(message, "result"):
        print(message.result)

print()
print("Archive hash unchanged:", sha256_of(ARCHIVE_PATH) == ARCHIVE_HASH)
show_files(WORKSHOP)

Workshop reset. Archive matches the original: True
Perfect! All three updates have been completed successfully. Here's a summary of the changes:

## Summary of Changes

1. **inventory.txt**: Updated sandpaper inventory from 40 sheets to 35 sheets (recorded the 5 sheets used)

2. **notes/todo.txt**: Added a new reminder line: "reorder wood glue"

3. **archive/2023-ledger.txt**: Fixed the spelling mistake from "wood glue botles used" to "wood glue bottles used"

All workshop records have been updated!

Archive hash unchanged: False
--- archive/2023-ledger.txt ---
Workshop Ledger 2023 (frozen record)
oak boards bought: 140
wood glue bottles used: 36
sandpaper sheets used: 520
tables finished: 18

--- inventory.txt ---
oak boards: 12
wood glue: 3
sandpaper sheets: 35
brass screws (box): 8

--- notes/todo.txt ---
sharpen the chisels
sweep the dust collector bin
label the offcut shelf
reorder wood glue

